# Load the data

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet("combined_new.parquet")
print(df.shape)
df.head()

In [ ]:
df["FL_DATE"] = pd.to_datetime(df["FL_DATE"])
df = df.drop_duplicates()
print(df.shape)


# Clean rows

Remove cancelled flights and rows where `DEP_DELAY` (our target source) is missing.

In [ ]:
df = df[df["CANCELLED"] == 0].copy()
print("After dropping cancellations:", df.shape)


In [ ]:
df = df.dropna(subset=["DEP_DELAY"])
print("After dropping missing target:", df.shape)


# Create multiclass target

Bin `DEP_DELAY` into 3 classes: No delay, moderate delay, and then severe delay

In [ ]:
def categorize_delay(minutes):
    if minutes < 15:    return 0  # No delay
    elif minutes < 91:  return 1  # 15–89 min
    else:               return 2 # 90+ minutes

df["DELAY_CLASS"] = df["DEP_DELAY"].apply(categorize_delay)
df["DELAY_CLASS"].value_counts(normalize=True).sort_index()

# Feature engineering

Extract state from `"City, ST"`, and extract `DEP_HOUR` (0–23) from the HHMM `CRS_DEP_TIME`.


In [ ]:
df["ORIGIN_STATE"] = df["ORIGIN_CITY_NAME"].str.split(",").str[-1].str.strip()
df["DEST_STATE"]   = df["DEST_CITY_NAME"].str.split(",").str[-1].str.strip()

df["DEP_HOUR"] = df["CRS_DEP_TIME"] // 100


#### Build PREV_FLIGHT_DELAY

For each flight, we look up the same aircraft's previous flight that day (via `TAIL_NUM`) and use that flight's arrival delay as a feature. If the plane landed late on its last leg, the next flight almost always departs late too — the plane just isn't at the gate yet.

This is the strongest pre-flight feature in the delay literature. It's not leakage: the previous flight landed before the current flight's scheduled departure.


In [ ]:
# Sort chronologically within each aircraft
df = df.sort_values(["TAIL_NUM", "FL_DATE", "CRS_DEP_TIME"]).reset_index(drop=True)

# Shift ARR_DELAY down by one row within each TAIL_NUM group
df["PREV_ARR_DELAY"] = df.groupby("TAIL_NUM")["ARR_DELAY"].shift(1)
df["PREV_FL_DATE"]   = df.groupby("TAIL_NUM")["FL_DATE"].shift(1)

# Only use the previous leg if it was on the same day (legitimate same-day turnaround)
same_day = df["PREV_FL_DATE"] == df["FL_DATE"]
df["PREV_FLIGHT_DELAY"] = np.where(same_day, df["PREV_ARR_DELAY"], np.nan)
df["HAS_PREV_FLIGHT"]   = same_day.astype(int)

# Fill NaN with 0 — first flight of the day for this aircraft, assume on-time start
df["PREV_FLIGHT_DELAY"] = df["PREV_FLIGHT_DELAY"].fillna(0)

df = df.drop(columns=["PREV_ARR_DELAY", "PREV_FL_DATE"])

print(f"HAS_PREV_FLIGHT=1: {df['HAS_PREV_FLIGHT'].mean():.1%} of flights")
print("PREV_FLIGHT_DELAY vs DELAY_CLASS (mean by bucket):")
df["_prev_bkt"] = pd.cut(
    df["PREV_FLIGHT_DELAY"],
    bins=[-np.inf, 15, 90, np.inf],
    right=False,
    labels=["no-delay", "moderate", "severe"],
)
print(df.groupby("_prev_bkt", observed=False)["DELAY_CLASS"].mean().round(3))
df = df.drop(columns=["_prev_bkt"])


In [ ]:
df["AIRPORT_TRAFFIC"] = df.groupby(["ORIGIN", "FL_DATE"])["OP_UNIQUE_CARRIER"].transform("nunique")

In [ ]:
from pandas.tseries.holiday import USFederalHolidayCalendar

cal = USFederalHolidayCalendar()
holidays = list(cal.holidays(start=df["FL_DATE"].min(), end=df["FL_DATE"].max()))

In [ ]:
from pandas.tseries.offsets import DateOffset
holiday_dates = pd.to_datetime(holidays)

adjacent = set()
for date in holiday_dates:
    adjacent.update([date - pd.Timedelta(days = 1),
                     date,
                     date + pd.Timedelta(days = 1)
                    ])

df["IS_HOLIDAY"] = df["FL_DATE"].isin(adjacent).astype(int)

# Drop columns we don't need

Drop columns only known after the flight, redundant/constant columns, and high-cardinality city columns (keeping state instead).

In [ ]:
drop_cols = [
    # used to build PREV_FLIGHT_DELAY, then dropped (leakage)
    "ARR_DELAY",
    # redundant with DEP_DELAY (just the positive-clipped version)
    "DEP_DELAY_NEW",
    # already filtered
    "CANCELLED",
    # redundant after feature construction
    "FL_DATE",
    "CRS_DEP_TIME",
    "ORIGIN_CITY_NAME",
    "DEST_CITY_NAME",
    # too many unique values for one-hot; already used to build PREV_FLIGHT_DELAY
    "TAIL_NUM",
    # drop airport codes — ORIGIN_STATE already captures location, fewer categories
    "ORIGIN",
    "DEST",
    # DEP_DELAY, DEP_DEL15, CARRIER_DELAY are kept for EDA and
    # dropped inside model_analysis.ipynb before training.
]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print(df.shape)
df.head()


# Save merged weather dataset

In [ ]:
from build_weather_dataset import main as build_weather_main

build_weather_main()

In [ ]:
df = pd.read_parquet('combined_preprocessed_weather.parquet')
print(df.shape)
df.head(30)